In [ ]:
#importing os library
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.chdir(r'C:\Users\ongja\Downloads\Intro_ML\Module1')

#reading Pos_Data and checking the first 5 rows
pos = pd.read_excel("pos_sample.xlsx")
pos.head()
#pos.tail()

,SKU_ID,Page_traffic,Units_sold,Segment
0,SKU1011,4119.0,697,Makeup
1,SKU1011,4074.0,1027,Makeup
2,SKU1011,4897.0,1037,Makeup
3,SKU1011,4783.0,1184,Makeup
4,SKU1011,3789.0,876,Makeup


In [4]:
# view the shape of the data
pos.shape

(3993, 4)

## Steps to Build a Prediction Model

1. Divide the data into train and validation.
2. Build mean-based and rule-based predictions.
3. Evaluate model correctness using MAE.

---

## Small Example (Mean Baseline + MAE)

**Train `Units_sold`**: 10, 12, 8, 10, 10  → **Mean = 10**

**Validation Actuals**: 9, 14, 7

| Row | Actual | Predicted (Mean) | Absolute Error |
|---|---:|---:|---:|
| 1 | 9 | 10 | 1 |
| 2 | 14 | 10 | 4 |
| 3 | 7 | 10 | 3 |

**MAE** = (1 + 4 + 3) / 3 = **2.67**

# Step 1: Divide the dataset

In [ ]:
#import train_test_split to split data into pos_train and pos_valid
from sklearn.model_selection import train_test_split

#split the data into pos_train and pos_valid|
pos_train, pos_valid= train_test_split(pos, test_size = 0.2, random_state = 9)

# 20% -> pos_valid & 80% -> pos_train

In [ ]:
#check the shape of train and validation data
pos_train.shape, pos_valid.shape

,SKU_ID,Page_traffic,Units_sold,Segment
2657,SKU469,1280.000000,787,Makeup
810,SKU1393,4151.000000,1103,Makeup
3783,SKU950,706.079631,340,Skincare
218,SKU1099,1712.000000,1053,Makeup
2223,SKU264,4871.000000,643,Skincare
1447,SKU157,4790.000000,1040,Skincare
1792,SKU1948,2200.000000,552,Skincare
930,SKU145,2311.000000,1105,Hair Care
2849,SKU481,2049.000000,889,Skincare
3678,SKU92,2306.000000,700,Hair Care


# Step 2: Perform Prediction (Mean Based Prediction)

In [9]:
#overall mean of Units_sold for train data and converting it into integers since units sold cannot be decimals
units_sold_mean = int(pos_train['Units_sold'].mean())

In [13]:
#creating a new column- "Pred_Mean_Units_Sold" in the pos_valid dataframe and assigning the mean of Units_sold from train data 
pos_valid['Pred_Mean_Units_Sold']= units_sold_mean
pos_valid.head()

,SKU_ID,Page_traffic,Units_sold,Segment,Pred_Mean_Units_Sold
2137,SKU252,3202.0,1303,Skincare,1157
118,SKU1056,4423.0,1586,Makeup,1157
903,SKU145,4562.0,1522,Hair Care,1157
1966,SKU199,4775.0,1495,Makeup,1157
2335,SKU271,2953.0,1138,Makeup,1157


# Step 3: Evaluate Prediction

In [ ]:
#import mean_absolute_error from sklearn.metrics
from sklearn.metrics import mean_absolute_error

#MAE of overall Units_sold mean from pos_train and actual values of 'Units_sold' from pos_valid
mean_absolute_error(pos_valid['Units_sold'], pos_valid['Pred_Mean_Units_Sold'])

307.0475594493116

# Approach 2 (Rule Based Prediction)

# Step 2: Perform Prediction

In [ ]:
# 1) Convert Unit_sold to numeric
pos_train['Units_sold'] = pd.to_numeric(pos_train['Units_sold'], errors='coerce')

# 2) Rule: Calculate Mean Unit sold for Train Dataset (pos_train) per segment.
mean_units_sold_by_segment = (
    pos_train.groupby('Segment')['Units_sold']
    .mean()
    .round(0)
    .astype('Int64')  # keeps NaNs if any
)
mean_units_sold_by_segment

Segment
Hair Care    1167
Makeup       1152
Skincare     1162
Name: Units_sold, dtype: Int64

In [ ]:
# Map those segment means onto pos_valid as predictions.
pos_valid['Pred_Segment_Mean_Units_Sold'] = pos_valid['Segment'].map(mean_units_sold_by_segment)
#view first 10 rows of pos_valid
pos_valid.head(10)

,SKU_ID,Page_traffic,Units_sold,Segment,Pred_Mean_Units_Sold,Pred_Segment_Mean_Units_Sold
2137,SKU252,3202.0,1303,Skincare,1157,1162
118,SKU1056,4423.0,1586,Makeup,1157,1152
903,SKU145,4562.0,1522,Hair Care,1157,1167
1966,SKU199,4775.0,1495,Makeup,1157,1152
2335,SKU271,2953.0,1138,Makeup,1157,1152
3339,SKU786,3216.0,1038,Makeup,1157,1152
595,SKU1266,4506.0,1125,Skincare,1157,1162
2241,SKU264,8062.0,1723,Skincare,1157,1162
310,SKU1099,2081.0,1247,Makeup,1157,1152
1987,SKU199,3274.0,1564,Makeup,1157,1152


# Step 3: Evaluate Prediction

In [ ]:
# Compute MAE between pos_valid['Units_sold'] and the predicted segment means.
mean_absolute_error(pos_valid['Units_sold'],pos_valid['Pred_Segment_Mean_Units_Sold'])

306.360450563204